In [1]:
import pandas as pd
import numpy as np

import torch
from torch import nn
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

In [2]:
import random

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

In [3]:
import geopandas as gpd
from libpysal.weights import Queen

gdf = gpd.read_file("../Output/Geo Output/London_LSOA_2021.shp").sort_values("LSOA21CD").reset_index(drop=True)

w = Queen.from_dataframe(gdf)
edges, weights = [], []

for src, neighbors in w.neighbors.items():
    for tgt in neighbors:
        geom_src = gdf.geometry.iloc[src]
        geom_tgt = gdf.geometry.iloc[tgt]
        inter = geom_src.intersection(geom_tgt)
        weight = inter.length if inter.length > 0 else 0.001
        edges.append([src, tgt])
        weights.append(weight)
        edges.append([tgt, src])
        weights.append(weight)
        
edge_index = torch.tensor(edges, dtype=torch.long).T
edge_weight = torch.tensor(weights, dtype=torch.float32)

C:\Users\wbwha\AppData\Local\Temp\ipykernel_29804\3828378298.py:6: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = Queen.from_dataframe(gdf)


In [4]:
X_train_sbert = torch.load("../Output/Pred Input/embedding_only_train_gcnlstm.pt")
X_train_sbert =X_train_sbert["X"]
X_train_topic = torch.load("../Output/Pred Input/topic_all_train_gcnlstm.pt")
X_train_topic =X_train_topic["X"]
X_train_topic50 = torch.load("../Output/Pred Input/topic_top50_train_gcnlstm.pt")
X_train_topic50 = X_train_topic50["X"]
X_train_topic100 = torch.load("../Output/Pred Input/topic_top100_train_gcnlstm.pt")
X_train_topic100 = X_train_topic100["X"]
X_train_topic200 = torch.load("../Output/Pred Input/topic_top200_train_gcnlstm.pt")
X_train_topic200 = X_train_topic200["X"]
X_train_topic500 = torch.load("../Output/Pred Input/topic_top500_train_gcnlstm.pt")
X_train_topic500 = X_train_topic500["X"]

y_train = torch.load("../Output/Pred Input/y_train_5.pt")

In [5]:
import os

from sklearn.metrics import f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

# =========================
# Candidates (sequence-only)
# =========================
# Will keep only tensors with shape [N, T, F] for LSTM.
candidates_raw = {
    "sbert":        X_train_sbert,
    "topic_all":    X_train_topic,
    "topic_top50":  X_train_topic50,
    "topic_top100": X_train_topic100,
    "topic_top200": X_train_topic200,
    "topic_top500": X_train_topic500,

}
candidates = {
    k: v for k, v in candidates_raw.items()
    if isinstance(v, torch.Tensor) and v.dim() == 3
}

# =========================
# Labels & device
# =========================
y = y_train  # keep your original label tensor
num_classes = int(len(torch.unique(y)))
assert y.dtype == torch.long, "y_train must be torch.long with class IDs 0..C-1"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# Model & training config
# =========================
from Hybrid_Models.Gen_GCN_LSTM_MLP import GentrificationGCN_LSTM

LR = 1e-3
WEIGHT_DECAY = 5e-4
MAX_EPOCHS = 100
PATIENCE = 50
LSTM_HIDDEN = 64
GCN_HIDDEN = 64
DROPOUT = 0.3

# Seeds (same style as before)
import random
random.seed(42)
SEEDS = random.sample(range(10000), 10)

# =========================
# Split utility (stratified)
# =========================
def stratified_split_indices(y_np: np.ndarray, val_ratio=0.2, seed=42):
    """Return train and val indices with class-stratified sampling."""
    train_idx, val_idx = [], []
    rng = np.random.RandomState(seed)
    for label in np.unique(y_np):
        idx = np.where(y_np == label)[0]
        n = len(idx); n_val = max(1, int(n * val_ratio))
        shuffled = rng.permutation(idx)
        val_idx.extend(shuffled[:n_val])
        train_idx.extend(shuffled[n_val:])
    return np.array(train_idx), np.array(val_idx)

# =========================
# One split train/eval
# =========================
def train_eval_one_split_seq(X_seq: torch.Tensor,
                             y: torch.Tensor,
                             edge_index,
                             edge_weight=None,
                             train_idx=None,
                             val_idx=None,
                             seed=42):
    """
    Train one GCN+LSTM instance and return validation metrics:
    - macro_f1, weighted_f1, accuracy
    Early stopping keyed on macro-F1.
    """
    # Reproducibility
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    # Move data
    X_seq = X_seq.to(device)            # [N, T, F]
    y_t   = y.to(device)
    ei    = edge_index.to(device)
    ew    = edge_weight.to(device) if edge_weight is not None else None

    # Define model (infer in_channels from X_seq.shape[2])
    in_channels = X_seq.shape[2]
    
    model = GentrificationGCN_LSTM(
        in_channels,        
        LSTM_HIDDEN,        
        GCN_HIDDEN,         
        DROPOUT,            
        num_classes      
    ).to(device)


    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = torch.nn.CrossEntropyLoss()  # keep same as your snippet

    best_macro = -1.0
    best_state = None
    patience = PATIENCE

    for epoch in range(1, MAX_EPOCHS + 1):
        # Train
        model.train()
        optimizer.zero_grad()
        out = model(X_seq, ei, ew)               # [N, C]
        loss = criterion(out[train_idx], y_t[train_idx])
        loss.backward()
        optimizer.step()

        # Validate
        model.eval()
        with torch.no_grad():
            logits_val = model(X_seq, ei, ew)[val_idx]
            y_pred = logits_val.argmax(dim=1).detach().cpu().numpy()
            y_true = y_t[val_idx].detach().cpu().numpy()
            macro_f1 = f1_score(y_true, y_pred, average="macro")

        if macro_f1 > best_macro:
            best_macro = macro_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience = PATIENCE
        else:
            patience -= 1
            if patience == 0:
                break

    # Load best and compute final val metrics
    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    model.eval()
    with torch.no_grad():
        logits_val = model(X_seq, ei, ew)[val_idx]
        y_pred = logits_val.argmax(dim=1).detach().cpu().numpy()
        y_true = y_t[val_idx].detach().cpu().numpy()

    metrics = {
        "macro_f1":    f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
        "accuracy":    accuracy_score(y_true, y_pred),
    }
    return metrics

# =========================
# Multi-seed evaluation
# =========================
def evaluate_dataset_multi_seed(name, X_seq, y, seeds=SEEDS, val_ratio=0.2):
    """
    For a given sequence feature set:
    - run multiple seeds,
    - collect macro_f1, weighted_f1, accuracy,
    - return per-metric mean and std (+ per-seed for traceability).
    """
    y_np = y.cpu().numpy()
    macro_list, weighted_list, acc_list = [], [], []
    per_seed = {"macro_f1": [], "weighted_f1": [], "accuracy": []}

    for s in seeds:
        tr_idx, va_idx = stratified_split_indices(y_np, val_ratio=val_ratio, seed=s)
        m = train_eval_one_split_seq(
            X_seq, y,
            edge_index=edge_index,
            edge_weight=edge_weight,
            train_idx=tr_idx, val_idx=va_idx, seed=s
        )
        macro_list.append(m["macro_f1"]); per_seed["macro_f1"].append(m["macro_f1"])
        weighted_list.append(m["weighted_f1"]); per_seed["weighted_f1"].append(m["weighted_f1"])
        acc_list.append(m["accuracy"]); per_seed["accuracy"].append(m["accuracy"])

    result = {
        "dataset": name,
        "n_features": int(X_seq.shape[2]),  # feature dim per time step
        "macro_f1_mean": float(np.mean(macro_list)),
        "macro_f1_std":  float(np.std(macro_list)),
        "weighted_f1_mean": float(np.mean(weighted_list)),
        "weighted_f1_std":  float(np.std(weighted_list)),
        "accuracy_mean": float(np.mean(acc_list)),
        "accuracy_std":  float(np.std(acc_list)),
        "per_seed": per_seed,
    }
    print(f"{name:12s} | F={X_seq.shape[2]:4d} | "
          f"macro={result['macro_f1_mean']:.4f}±{result['macro_f1_std']:.4f}  "
          f"weighted={result['weighted_f1_mean']:.4f}±{result['weighted_f1_std']:.4f}  "
          f"acc={result['accuracy_mean']:.4f}±{result['accuracy_std']:.4f}")
    return result

# =========================
# Run comparison & save
# =========================
results = []
for name, X_seq in candidates.items():
    # Quick sanity checks
    assert X_seq.dim() == 3, f"{name} must be [N, T, F]"
    assert X_seq.shape[0] == y.shape[0], f"{name}: N mismatch between X and y"
    results.append(evaluate_dataset_multi_seed(name, X_seq, y, seeds=SEEDS, val_ratio=0.2))

# Sort by primary metric (macro F1 mean) descending
results_sorted = sorted(results, key=lambda d: d["macro_f1_mean"], reverse=True)

# Build DataFrame for summary
import pandas as pd
df = pd.DataFrame([
    {
        "dataset": r["dataset"],
        "n_features": r["n_features"],
        "macro_f1_mean": r["macro_f1_mean"],
        "macro_f1_std": r["macro_f1_std"],
        "weighted_f1_mean": r["weighted_f1_mean"],
        "weighted_f1_std": r["weighted_f1_std"],
        "accuracy_mean": r["accuracy_mean"],
        "accuracy_std": r["accuracy_std"],
    }
    for r in results_sorted
])

# Ensure output directory exists and save
out_dir = "../Output/Performance"
os.makedirs(out_dir, exist_ok=True)
csv_summary = os.path.join(out_dir, "gcn_lstm_dataset_comparison.csv")
csv_long = os.path.join(out_dir, "gcn_lstm_dataset_per_seed_long.csv")

df.to_csv(csv_summary, index=False)

# Also save a long-form per-seed CSV for traceability
rows = []
for r in results_sorted:
    ds = r["dataset"]
    for i, s in enumerate(SEEDS):
        rows.append({
            "dataset": ds,
            "seed": s,
            "macro_f1": r["per_seed"]["macro_f1"][i],
            "weighted_f1": r["per_seed"]["weighted_f1"][i],
            "accuracy": r["per_seed"]["accuracy"][i],
        })
pd.DataFrame(rows).to_csv(csv_long, index=False)

print("\n=== Ranking (by macro F1 mean) ===")
print(df[["dataset", "n_features", "macro_f1_mean", "macro_f1_std",
          "weighted_f1_mean", "weighted_f1_std", "accuracy_mean", "accuracy_std"]])
print(f"\nSaved summary to: {csv_summary}")
print(f"Saved per-seed to: {csv_long}")

sbert        | F= 384 | macro=0.3319±0.1246  weighted=0.4929±0.1079  acc=0.5382±0.0699
topic_all    | F=1174 | macro=0.4212±0.0181  weighted=0.5637±0.0164  acc=0.5769±0.0191
topic_top50  | F=  50 | macro=0.3841±0.0174  weighted=0.5407±0.0121  acc=0.5668±0.0086
topic_top100 | F= 100 | macro=0.4111±0.0129  weighted=0.5570±0.0120  acc=0.5681±0.0180
topic_top200 | F= 200 | macro=0.4203±0.0099  weighted=0.5686±0.0079  acc=0.5869±0.0101
topic_top500 | F= 500 | macro=0.4197±0.0112  weighted=0.5645±0.0094  acc=0.5759±0.0129

=== Ranking (by macro F1 mean) ===
        dataset  n_features  macro_f1_mean  macro_f1_std  weighted_f1_mean  \
0     topic_all        1174       0.421235      0.018058          0.563720   
1  topic_top200         200       0.420339      0.009932          0.568584   
2  topic_top500         500       0.419693      0.011242          0.564537   
3  topic_top100         100       0.411129      0.012865          0.557042   
4   topic_top50          50       0.384122      0.01

In [7]:
import os, itertools, random

from sklearn.metrics import f1_score, accuracy_score, recall_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from Loss_Functions.Focal_Loss import FocalLossMultiClass

# ------------------------------------------------------------
# Inputs assumed available:
#   X_train_topic200: [N, T, F] tensor on device (timeseries features)
#   y_train:          [N] long tensor
#   edge_index, edge_weight
# ------------------------------------------------------------

device = X_train_topic200.device if X_train_topic200.is_cuda else ("cuda" if torch.cuda.is_available() else "cpu")
X_train_topic200 = X_train_topic200.to(device)
y_train = y_train.to(device)
n_classes = int(len(torch.unique(y_train)))
in_channels = int(X_train_topic200.shape[2])

# Fixed 5-fold CV over nodes
y_np = y_train.cpu().numpy()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_indices = list(skf.split(np.arange(len(y_np)), y_np))

# Search space for GCN+LSTM (note: no focal_gamma here; gamma comes from loss variants below)
search_space = {
    "lstm_hidden":  [64, 128],
    "gcn_hidden":   [64, 128],
    "dropout":      [0.1, 0.3],
    "lr":           [1e-3, 5e-4],
    "weight_decay": [5e-4, 1e-4],
}
def product_dict(d):
    keys = list(d.keys())
    for values in itertools.product(*[d[k] for k in keys]):
        yield dict(zip(keys, values))

# Loss variants (from script 1 logic)
def compute_class_weights_on_train(y_np_train, num_classes, device):
    classes_present = np.unique(y_np_train)
    weights_partial = compute_class_weight(class_weight='balanced', classes=classes_present, y=y_np_train)
    full = np.zeros(num_classes, dtype=np.float32)
    for i, cls in enumerate(classes_present):
        full[int(cls)] = weights_partial[i]
    if (full == 0).any():
        nz = full[full > 0]
        full[full == 0] = nz.min() if len(nz) > 0 else 1.0
    return torch.tensor(full, dtype=torch.float32, device=device)

loss_variants = {
    "CE":           lambda w: torch.nn.CrossEntropyLoss(),
    "Focal_gamma1": lambda w: FocalLossMultiClass(alpha=w, gamma=1.0),
    "Focal_gamma2": lambda w: FocalLossMultiClass(alpha=w, gamma=2.0),
}

# Training control
seeds = [0, 1, 2]
num_epochs = 100
eval_every = 5
early_stop_patience = 50
early_stop_min_delta = 1e-4
monitor_metric = "val_macro_f1"

def eval_on_mask(logits, y_true_tensor, mask):
    y_true = y_true_tensor[mask].cpu().numpy()
    y_pred = logits[mask].argmax(dim=1).cpu().numpy()
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    rec_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
    return acc, f1_macro, f1_weighted, rec_macro, y_true, y_pred

def train_one_run(cfg, loss_name, tr_idx, va_idx, seed):
    tr_mask = torch.zeros(len(y_train), dtype=torch.bool, device=device); tr_mask[tr_idx] = True
    va_mask = torch.zeros(len(y_train), dtype=torch.bool, device=device); va_mask[va_idx] = True

    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    # Model
    model = GentrificationGCN_LSTM(
        in_channels,
        cfg["lstm_hidden"],
        cfg["gcn_hidden"],
        cfg["dropout"],
        n_classes
    ).to(device)

    # Optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    # Loss with per-fold class weights
    class_w = compute_class_weights_on_train(y_np[tr_idx], n_classes, device)
    criterion = loss_variants[loss_name](class_w)

    best_metric = -1.0
    best_state = None
    best_epoch = 0
    no_improve = 0

    history = {"epoch": [], "train_loss": [], "val_loss": [], "val_acc": [], "val_macro_f1": [],
               "val_weighted_f1": [], "val_macro_recall": []}

    for epoch in range(1, num_epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(X_train_topic200, edge_index, edge_weight)  # [N, C]
        loss = criterion(out[tr_mask], y_train[tr_mask])
        loss.backward()
        optimizer.step()

        if (epoch % eval_every == 0) or (epoch == 1) or (epoch == num_epochs):
            model.eval()
            with torch.no_grad():
                out_eval = model(X_train_topic200, edge_index, edge_weight)
                val_logits = out_eval[va_mask]
                val_loss = criterion(val_logits, y_train[va_mask]).item()

                acc, f1m, f1w, recm, _, _ = eval_on_mask(out_eval, y_train, va_mask)

            history["epoch"].append(epoch)
            history["train_loss"].append(float(loss.item()))
            history["val_loss"].append(float(val_loss))
            history["val_acc"].append(float(acc))
            history["val_macro_f1"].append(float(f1m))
            history["val_weighted_f1"].append(float(f1w))
            history["val_macro_recall"].append(float(recm))

            current = f1m if monitor_metric == "val_macro_f1" else acc
            improved = (current - best_metric) > early_stop_min_delta
            if improved:
                best_metric = current
                best_epoch = epoch
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= early_stop_patience:
                    break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        out_final = model(X_train_topic200, edge_index, edge_weight)
        acc, f1m, f1w, recm, y_true, y_pred = eval_on_mask(out_final, y_train, va_mask)
        cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))
        report = classification_report(y_true, y_pred, digits=4, zero_division=0)

    return {
        "final_acc": acc, "final_macro_f1": f1m, "final_weighted_f1": f1w,
        "final_macro_recall": recm, "best_epoch": best_epoch,
        "history": history, "report": report, "confusion_matrix": cm
    }

# Run grid × loss × (5 folds) × seeds
results = []
for cfg in product_dict(search_space):
    for loss_name in ["CE", "Focal_gamma1", "Focal_gamma2"]:
        fold_metrics = []
        rep_history = None; rep_report = None; rep_cm = None

        for fold_id, (tr_idx, va_idx) in enumerate(fold_indices, 1):
            seed_metrics = []
            for s in seeds:
                m = train_one_run(cfg, loss_name, tr_idx, va_idx, seed=s)
                seed_metrics.append(m)

            if rep_history is None:
                rep_history = seed_metrics[0]["history"]
                rep_report = seed_metrics[0]["report"]
                rep_cm = seed_metrics[0]["confusion_matrix"]

            fold_metrics.append({
                "acc": np.mean([x["final_acc"] for x in seed_metrics]),
                "macro_f1": np.mean([x["final_macro_f1"] for x in seed_metrics]),
                "weighted_f1": np.mean([x["final_weighted_f1"] for x in seed_metrics]),
                "macro_recall": np.mean([x["final_macro_recall"] for x in seed_metrics]),
                "best_epoch": np.mean([x["best_epoch"] for x in seed_metrics]),
            })

        avg_acc = float(np.mean([fm["acc"] for fm in fold_metrics]))
        avg_macro = float(np.mean([fm["macro_f1"] for fm in fold_metrics]))
        avg_weighted = float(np.mean([fm["weighted_f1"] for fm in fold_metrics]))
        avg_recall = float(np.mean([fm["macro_recall"] for fm in fold_metrics]))
        avg_best_epoch = int(np.round(np.mean([fm["best_epoch"] for fm in fold_metrics])))

        results.append({
            "cfg": cfg, "loss": loss_name,
            "avg_acc": avg_acc, "avg_macro_f1": avg_macro,
            "avg_weighted_f1": avg_weighted, "avg_macro_recall": avg_recall,
            "avg_best_epoch": avg_best_epoch,
            "history": rep_history, "report_seed0": rep_report, "cm_seed0": rep_cm
        })

# Save leaderboard CSV
os.makedirs("../Output/Performance", exist_ok=True)
df = pd.DataFrame([{
    **r["cfg"], "loss": r["loss"],
    "avg_macro_f1": r["avg_macro_f1"],
    "avg_weighted_f1": r["avg_weighted_f1"],
    "avg_accuracy": r["avg_acc"],
    "avg_macro_recall": r["avg_macro_recall"],
    "avg_best_epoch": r["avg_best_epoch"],
    "n_folds": 5, "seeds": len(seeds),
} for r in results])

df = df.sort_values(by=["avg_macro_f1", "avg_weighted_f1", "avg_accuracy"], ascending=False).reset_index(drop=True)
csv_path = "../Output/Performance/gcn_lstm_5fold_grid_results.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"Saved: {csv_path}")
print(df.head(10))

Saved: ../Output/Performance/gcn_lstm_5fold_grid_results.csv
   lstm_hidden  gcn_hidden  dropout      lr  weight_decay          loss  \
0           64         128      0.3  0.0005        0.0001            CE   
1           64         128      0.3  0.0005        0.0005            CE   
2           64         128      0.3  0.0010        0.0005            CE   
3           64         128      0.3  0.0010        0.0001            CE   
4          128         128      0.3  0.0005        0.0005            CE   
5          128         128      0.3  0.0005        0.0001  Focal_gamma1   
6          128         128      0.3  0.0005        0.0001            CE   
7          128         128      0.3  0.0005        0.0001  Focal_gamma2   
8          128         128      0.3  0.0005        0.0005  Focal_gamma1   
9           64          64      0.3  0.0010        0.0001            CE   

   avg_macro_f1  avg_weighted_f1  avg_accuracy  avg_macro_recall  \
0      0.418963         0.568862      0.58183